# Türkçe Yorum Sentiment Analizi — Colab Eğitim Notebook'u

Bu notebook, HepsiBurada ürün yorumlarını **pozitif** / **negatif** olarak sınıflandıran bir BERT modeli eğitir.

**Model:** `dbmdz/bert-base-turkish-cased`

**Veri:** Elle etiketlenmiş `etiketleme.csv` dosyası kullanılır.

**Adımlar:**
1. Kütüphane kurulumu
2. Veriyi yükleme ve inceleme
3. Metin temizleme
4. Tokenizer testi
5. Dataset oluşturma
6. Modeli eğitme (ağırlıklı loss ile)
7. Değerlendirme (F1, Confusion Matrix)
8. Tüm ürünler için JSON çıktısı üretme

> **Not:** Runtime → Change runtime type → **T4 GPU** seçili olmalı.

## 1. Kütüphane Kurulumu

In [ ]:
!pip install -q transformers torch pandas scikit-learn matplotlib accelerate

## 2. Veriyi Yükleme ve İnceleme

İki dosya yüklemeniz gerekiyor:
1. `etiketleme.csv` — Elle etiketlediğiniz dosya (eğitim verisi)
2. `user_contents.csv` — Scraper'ın ürettiği ham CSV (tahmin aşaması için)

In [ ]:
import pandas as pd
from google.colab import files

# ── Dosyaları yükle ──
print("Lütfen etiketleme.csv ve user_contents.csv dosyalarını yükleyin:")
yuklenen = files.upload()

# ── Elle etiketlenmiş veriyi oku ──
df = pd.read_csv("etiketleme.csv", encoding="utf-8")

# Sadece label sütunu dolu olan satırları al
df = df.dropna(subset=["label"])
df = df[df["label"].astype(str).str.strip() != ""]
df["label"] = df["label"].astype(int)
df = df[df["label"].isin([0, 1])]

print(f"\nToplam etiketli yorum: {len(df)}")
print(f"\nSınıf dağılımı:")
print(df["label"].value_counts().rename({0: "Negatif (0)", 1: "Pozitif (1)"}))
print(f"\nİlk 5 yorum:")
df[["comment", "star", "label"]].head(10)

## 3. Metin Temizleme ve Train/Test Split

In [ ]:
import re
from sklearn.model_selection import train_test_split


def metin_temizle(metin: str) -> str:
    """Emoji ve özel karakterleri temizler, fazla boşlukları siler."""
    if not isinstance(metin, str):
        return ""
    metin = metin.replace("\n", " ").replace("\r", " ")
    metin = re.sub(r"[^\w\s.,!?;:'\"-()/ ]", " ", metin)
    metin = re.sub(r"\s+", " ", metin).strip()
    return metin


# ── Metni temizle ──
df["comment"] = df["comment"].apply(metin_temizle)
df = df[df["comment"].str.strip().astype(bool)]

# ── Train / Test split ──
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)},  Test: {len(test_df)}")
print(f"\nTrain dağılımı:")
print(train_df["label"].value_counts().rename({0: "Negatif", 1: "Pozitif"}))

## 4. Tokenizer Testi
`dbmdz/bert-base-turkish-cased` tokenizer'ını yükleyip örnek bir cümleyi tokenize edelim.

In [ ]:
from transformers import BertTokenizer

MODEL_ADI = "dbmdz/bert-base-turkish-cased"
MAX_UZUNLUK = 128

tokenizer = BertTokenizer.from_pretrained(MODEL_ADI)

# Örnek bir cümle
ornek = "Ürün çok güzel ama kargo geç geldi."
kodlama = tokenizer(ornek, max_length=MAX_UZUNLUK, padding="max_length", truncation=True)

print(f"Orijinal metin : {ornek}")
print(f"Token sayısı   : {sum(kodlama['attention_mask'])}")
print(f"Token ID'leri  : {kodlama['input_ids'][:20]}...")
print(f"Tokenler       : {tokenizer.convert_ids_to_tokens(kodlama['input_ids'][:20])}")

## 5. Dataset Oluşturma

In [ ]:
import torch
from torch.utils.data import Dataset


class YorumDataset(Dataset):
    """Yorumları BERT girdisine dönüştüren PyTorch Dataset."""

    def __init__(self, metinler, etiketler, tokenizer, max_uzunluk=128):
        self.metinler = metinler
        self.etiketler = etiketler
        self.tokenizer = tokenizer
        self.max_uzunluk = max_uzunluk

    def __len__(self):
        return len(self.metinler)

    def __getitem__(self, idx):
        kodlama = self.tokenizer(
            str(self.metinler[idx]),
            max_length=self.max_uzunluk,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": kodlama["input_ids"].squeeze(0),
            "attention_mask": kodlama["attention_mask"].squeeze(0),
            "labels": torch.tensor(int(self.etiketler[idx]), dtype=torch.long),
        }


# ── Dataset nesneleri ──
train_dataset = YorumDataset(
    metinler=train_df["comment"].tolist(),
    etiketler=train_df["label"].tolist(),
    tokenizer=tokenizer,
)
test_dataset = YorumDataset(
    metinler=test_df["comment"].tolist(),
    etiketler=test_df["label"].tolist(),
    tokenizer=tokenizer,
)

ornek = train_dataset[0]
print(f"input_ids shape    : {ornek['input_ids'].shape}")
print(f"attention_mask shape: {ornek['attention_mask'].shape}")
print(f"label              : {ornek['labels']}")

## 6. Modeli Eğitme (Ağırlıklı Loss ile)

Sınıf dengesizliğini çözmek için **ağırlıklı CrossEntropyLoss** kullanıyoruz.
Azınlık sınıfına (negatif) daha fazla ağırlık verilir.

- **Epoch:** 5
- **Batch size:** 16
- **Learning rate:** 2e-5

In [ ]:
import torch.nn as nn
import numpy as np
from transformers import (
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


# ── Sınıf ağırlıklarını hesapla ──
etiketler = train_df["label"].tolist()
toplam = len(etiketler)
n_neg = etiketler.count(0)
n_pos = etiketler.count(1)
agirliklar = [toplam / (2 * n_neg), toplam / (2 * n_pos)]
print(f"Sınıf ağırlıkları → Negatif: {agirliklar[0]:.2f}, Pozitif: {agirliklar[1]:.2f}")


class DengeliTrainer(Trainer):
    """Sınıf dengesizliğini çözen özel Trainer."""

    def __init__(self, sinif_agirliklari=None, **kwargs):
        super().__init__(**kwargs)
        self.sinif_agirliklari = sinif_agirliklari

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        if self.sinif_agirliklari is not None:
            weight = torch.tensor(
                self.sinif_agirliklari, dtype=torch.float32
            ).to(logits.device)
            loss_fn = nn.CrossEntropyLoss(weight=weight)
        else:
            loss_fn = nn.CrossEntropyLoss()

        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


def metrikleri_hesapla(eval_pred):
    """Her epoch sonunda accuracy, F1, precision, recall hesaplar."""
    tahminler, gercekler = eval_pred
    tahmin_etiketleri = np.argmax(tahminler, axis=1)
    return {
        "accuracy": accuracy_score(gercekler, tahmin_etiketleri),
        "f1": f1_score(gercekler, tahmin_etiketleri, average="binary"),
        "precision": precision_score(gercekler, tahmin_etiketleri, average="binary"),
        "recall": recall_score(gercekler, tahmin_etiketleri, average="binary"),
    }


# ── Model ──
model = BertForSequenceClassification.from_pretrained(MODEL_ADI, num_labels=2)

# ── Eğitim ayarları ──
egitim_ayarlari = TrainingArguments(
    output_dir="./sonuclar/model",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=10,
    seed=42,
    report_to="none",
)

trainer = DengeliTrainer(
    sinif_agirliklari=agirliklar,
    model=model,
    args=egitim_ayarlari,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=metrikleri_hesapla,
)

# ── Eğitimi başlat ──
print("\nEğitim başlıyor...")
trainer.train()
print("Eğitim tamamlandı!")

## 7. Değerlendirme (F1, Confusion Matrix)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# ── Test seti tahmini ──
tahmin_sonucu = trainer.predict(test_dataset)
tahmin_etiketleri = np.argmax(tahmin_sonucu.predictions, axis=1)
gercek_etiketler = test_df["label"].values

# ── Classification Report ──
print("=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(
    classification_report(
        gercek_etiketler,
        tahmin_etiketleri,
        target_names=["Negatif (0)", "Pozitif (1)"],
    )
)

# ── Confusion Matrix ──
cm = confusion_matrix(gercek_etiketler, tahmin_etiketleri)
gorsel = ConfusionMatrixDisplay(cm, display_labels=["Negatif", "Pozitif"])
gorsel.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 7b. Modeli Kaydetme

In [ ]:
kayit_yolu = "./sonuclar/model/best_model"
trainer.save_model(kayit_yolu)
tokenizer.save_pretrained(kayit_yolu)
print(f"Model kaydedildi → {kayit_yolu}")

## 8. Tüm Ürünler İçin Konu Bazlı Analiz

Her yorumu cümlelere bölüp sınıflandırıyoruz, ardından **konu bazlı gruplama** ve **özetleme** yapıyoruz.

Çıktı formatı:
```json
{
  "HBCV00005QG5TV": {
    "ozet": { "toplam_yorum": 319, "pozitif_cumle": 207, "negatif_cumle": 257 },
    "artilar": [{ "baslik": "Ürünün görünümü beğeniliyor.", "sayi": 89 }],
    "eksiler": [{ "baslik": "Malzemede kırılma hataları var.", "sayi": 49 }],
    "tartismali": [{
      "baslik": "Orijinallik konusunda görüşler bölünmüş.",
      "detay": "34 kullanıcı olumlu bulurken, 92 kullanıcı olumsuz buluyor.",
      "pozitif": 34, "negatif": 92
    }]
  }
}
```

In [ ]:
import json
import os
from collections import defaultdict

# ── Orijinal CSV'yi yükle (tüm yorumlar) ──
df_tum = pd.read_csv("user_contents.csv", encoding="utf-8")

cihaz = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(cihaz)
model.eval()

MIN_KELIME = 3
GUVEN_ESIGI = 0.6
TARTISMA_ESIGI = 0.30

# ── Konu tanımları ──
KONULAR = {
    "kalite": {
        "kelimeler": ["kalite", "kaliteli", "kalitesiz", "sağlam", "dayanıklı", "dandik"],
        "pozitif": "Ürün kalitesi beğeniliyor.",
        "negatif": "Ürün kalitesi beğenilmiyor.",
        "tartismali": "Kalite konusunda görüşler bölünmüş.",
    },
    "orijinallik": {
        "kelimeler": ["orijinal", "orjinal", "sahte", "çakma", "taklit", "imitasyon",
                       "imtasyon", "emitasyon", "barkod", "barkot", "qr", "karekod"],
        "pozitif": "Kullanıcılar ürünün orijinal olduğunu düşünüyor.",
        "negatif": "Kullanıcılar ürünün orijinal olmadığını düşünüyor.",
        "tartismali": "Orijinallik konusunda görüşler bölünmüş.",
    },
    "beden_kalip": {
        "kelimeler": ["dar", "geniş", "büyük", "küçük", "tam kalıp", "kalıp",
                       "numara", "beden", "sıkı", "sıktı", "taraklı", "buçuk"],
        "pozitif": "Beden/kalıp uygun bulunuyor.",
        "negatif": "Beden/kalıp sorunlu bulunuyor, farklı numara öneriliyor.",
        "tartismali": "Beden/kalıp konusunda görüşler farklılık gösteriyor.",
    },
    "kargo_paketleme": {
        "kelimeler": ["kargo", "paketleme", "paketlenme", "kutu", "kutusuz",
                       "ezilmiş", "kırık", "yırtık", "poşet", "teslimat", "teslim"],
        "pozitif": "Kargo ve paketleme beğeniliyor.",
        "negatif": "Kargo ve paketleme konusunda ciddi şikayetler var.",
        "tartismali": "Kargo/paketleme konusunda görüşler bölünmüş.",
    },
    "fiyat": {
        "kelimeler": ["fiyat", "ucuz", "pahalı", "uygun", "indirim", "fiyat performans", "para"],
        "pozitif": "Fiyat-performans oranı beğeniliyor.",
        "negatif": "Fiyatı yüksek bulunuyor.",
        "tartismali": "Fiyat konusunda görüşler bölünmüş.",
    },
    "rahatlik": {
        "kelimeler": ["rahat", "rahatsız", "vuruyor", "vurdu", "sıkıyor",
                       "konforlu", "ağrı", "acıtı", "kanattı"],
        "pozitif": "Ürün rahat bulunuyor.",
        "negatif": "Ürün rahatsız bulunuyor, ayağı vurduğu belirtiliyor.",
        "tartismali": "Rahatlık konusunda görüşler bölünmüş.",
    },
    "gorunum": {
        "kelimeler": ["şık", "güzel", "tatlı", "hoş", "harika", "mükemmel",
                       "çirkin", "model", "tasarım", "görünüm", "duruş"],
        "pozitif": "Ürünün görünümü ve tasarımı beğeniliyor.",
        "negatif": "Ürünün görünümü beklentiyi karşılamıyor.",
        "tartismali": "Görünüm konusunda görüşler bölünmüş.",
    },
    "koku": {
        "kelimeler": ["koku", "kokuyor", "bali", "yapıştırıcı koku", "plastik koku", "leş"],
        "pozitif": "Üründe koku sorunu yaşanmamış.",
        "negatif": "Üründe kötü koku şikayetleri var.",
        "tartismali": "Koku konusunda görüşler bölünmüş.",
    },
    "malzeme": {
        "kelimeler": ["deri", "kırılma", "kırışık", "kırıldı", "soyulma",
                       "yırtıl", "yamuk", "defolu", "defo", "yapıştırıcı iz", "dikiş"],
        "pozitif": "Malzeme kalitesi beğeniliyor.",
        "negatif": "Malzemede kırılma, kırışma veya dikiş hataları görülüyor.",
        "tartismali": "Malzeme kalitesi konusunda görüşler bölünmüş.",
    },
    "iade": {
        "kelimeler": ["iade", "değişim", "geri gönderdim", "geri yolladım"],
        "pozitif": "İade/değişim süreci sorunsuz işlemiş.",
        "negatif": "İade veya değişim talebi oluşmuş.",
        "tartismali": "İade süreçleri konusunda görüşler bölünmüş.",
    },
}


def cumlelere_bol(metin):
    cumleler = re.split(r"(?<=[.!?])\s+", metin)
    if len(cumleler) == 1 and len(metin.split()) > 10:
        cumleler = re.split(r"(?<=,)\s+", metin)
    sonuc = [c.strip() for c in cumleler if len(c.strip().split()) >= MIN_KELIME]
    return sonuc if sonuc else [metin]


def tahmin_yap(metin):
    kodlama = tokenizer(metin, max_length=128, padding="max_length",
                         truncation=True, return_tensors="pt")
    ids = kodlama["input_ids"].to(cihaz)
    mask = kodlama["attention_mask"].to(cihaz)
    with torch.no_grad():
        cikti = model(input_ids=ids, attention_mask=mask)
    probs = torch.softmax(cikti.logits, dim=1)
    etiket = torch.argmax(probs, dim=1).item()
    guven = probs[0][etiket].item()
    return etiket, guven


def konu_tespit_et(cumle):
    cumle_kucuk = cumle.lower()
    bulunan = []
    for konu_adi, bilgi in KONULAR.items():
        for kelime in bilgi["kelimeler"]:
            if kelime.lower() in cumle_kucuk:
                bulunan.append(konu_adi)
                break
    return bulunan if bulunan else ["genel"]


def ozetler_uret(konu_sayilari):
    artilar, eksiler, tartismali = [], [], []
    for konu, sayilar in sorted(konu_sayilari.items(),
                                 key=lambda x: x[1]["pozitif"]+x[1]["negatif"], reverse=True):
        poz, neg = sayilar["pozitif"], sayilar["negatif"]
        toplam = poz + neg
        if toplam < 2:
            continue
        bilgi = KONULAR.get(konu)
        if konu == "genel":
            if poz > neg:
                artilar.append({"baslik": "Genel olarak ürün beğeniliyor.", "sayi": poz})
            elif neg > poz:
                eksiler.append({"baslik": "Genel olarak ürün beğenilmiyor.", "sayi": neg})
            continue
        azinlik = min(poz, neg)
        cogunluk = max(poz, neg)
        oran = azinlik / cogunluk if cogunluk > 0 else 0
        if oran >= TARTISMA_ESIGI and azinlik >= 3:
            tartismali.append({"baslik": bilgi["tartismali"],
                                "detay": f"{poz} kullanıcı olumlu bulurken, {neg} kullanıcı olumsuz buluyor.",
                                "pozitif": poz, "negatif": neg})
        elif poz >= neg:
            artilar.append({"baslik": bilgi["pozitif"], "sayi": poz})
        else:
            eksiler.append({"baslik": bilgi["negatif"], "sayi": neg})
    return {"artilar": artilar, "eksiler": eksiler, "tartismali": tartismali}


# ── Analiz ──
sonuc = {}

for urun_id in df_tum["item"].unique():
    yorumlar = df_tum[df_tum["item"] == urun_id]["comment"].tolist()
    konu_sayilari = defaultdict(lambda: {"pozitif": 0, "negatif": 0})
    toplam_poz, toplam_neg = 0, 0

    for yorum in yorumlar:
        yorum_temiz = metin_temizle(str(yorum))
        if not yorum_temiz:
            continue
        for cumle in cumlelere_bol(yorum_temiz):
            etiket, guven = tahmin_yap(cumle)
            if guven < GUVEN_ESIGI:
                continue
            key = "pozitif" if etiket == 1 else "negatif"
            for konu in konu_tespit_et(cumle):
                konu_sayilari[konu][key] += 1
            if etiket == 1:
                toplam_poz += 1
            else:
                toplam_neg += 1

    ozetler = ozetler_uret(dict(konu_sayilari))
    sonuc[urun_id] = {
        "ozet": {"toplam_yorum": len(yorumlar),
                 "pozitif_cumle": toplam_poz, "negatif_cumle": toplam_neg},
        **ozetler,
    }
    print(f"{urun_id}: {len(yorumlar)} yorum → "
          f"{len(ozetler['artilar'])} artı, {len(ozetler['eksiler'])} eksi, "
          f"{len(ozetler['tartismali'])} tartışmalı")

# ── JSON kaydet ──
os.makedirs("./sonuclar", exist_ok=True)
with open("./sonuclar/urun_analiz.json", "w", encoding="utf-8") as f:
    json.dump(sonuc, f, ensure_ascii=False, indent=2)
print("\nJSON kaydedildi → ./sonuclar/urun_analiz.json")

## Sonuç

In [ ]:
for urun_id, veri in sonuc.items():
    oz = veri["ozet"]
    print(f"\n{'='*60}")
    print(f"  Ürün: {urun_id}")
    print(f"  Toplam: {oz['toplam_yorum']} yorum  |  "
          f"+{oz['pozitif_cumle']} pozitif  / -{oz['negatif_cumle']} negatif cümle")
    print(f"{'='*60}")

    print(f"\n  ✅ GÜÇLÜ YÖNLER ({len(veri['artilar'])})")
    for i, a in enumerate(veri["artilar"], 1):
        print(f"    {i}. {a['baslik']}  ({a['sayi']} kullanıcı)")

    print(f"\n  ❌ ZAYIF YÖNLER ({len(veri['eksiler'])})")
    for i, e in enumerate(veri["eksiler"], 1):
        print(f"    {i}. {e['baslik']}  ({e['sayi']} kullanıcı)")

    if veri["tartismali"]:
        print(f"\n  ⚖️  TARTIŞMALI KONULAR ({len(veri['tartismali'])})")
        for i, t in enumerate(veri["tartismali"], 1):
            print(f"    {i}. {t['baslik']}")
            print(f"       {t['detay']}")

## İndirme
Eğitilmiş modeli ve JSON çıktısını bilgisayarınıza indirin.

In [ ]:
# JSON çıktısını indir
files.download("./sonuclar/urun_analiz.json")

# Modeli zip'le ve indir
!zip -r sonuclar_model.zip ./sonuclar/model/best_model/
files.download("sonuclar_model.zip")

print("İndirme tamamlandı!")
print()
print("Lokalde kullanmak için:")
print("  1. sonuclar_model.zip dosyasını proje klasörüne koy")
print("  2. unzip sonuclar_model.zip")
print("  3. streamlit run app.py")
print("  4. 'Yeni Ürün Analizi' sekmesinden herhangi bir ürün ID'si gir")